<a href="https://colab.research.google.com/github/gangambikaavanti/ai_lab_programs/blob/main/wee8_1BM24CS101.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
class ForwardChainingKB:
    def __init__(self):
        # Initial known facts
        self.facts = set([
            ("American", "Robert"),
            ("Enemy", "A", "America"),
            ("Owns", "A", "T1"),
            ("Missile", "T1")
        ])

        # Rules defined as functions that check facts and derive new ones
        self.rules = [
            self.rule_missile_is_weapon,
            self.rule_missile_sold_by_robert,
            self.rule_enemy_is_hostile,
            self.rule_criminal_definition
        ]

    # Rule 1: Missile(x) => Weapon(x)
    def rule_missile_is_weapon(self):
        new_facts = set()
        for fact in list(self.facts):
            if fact[0] == "Missile":
                x = fact[1]
                new_fact = ("Weapon", x)
                if new_fact not in self.facts:
                    print(f"-> Derived: Weapon({x}) via Missile({x}) => Weapon({x})")
                    new_facts.add(new_fact)
        return new_facts

    # Rule 2: Missile(x) ∧ Owns(A, x) => Sells(Robert, x, A)
    def rule_missile_sold_by_robert(self):
        new_facts = set()
        missiles = {f[1] for f in self.facts if f[0] == "Missile"}
        owned_by_A = {f[2] for f in self.facts if f[0] == "Owns" and f[1] == "A"}

        # Find intersection where x is a missile AND owned by A
        for x in missiles.intersection(owned_by_A):
            new_fact = ("Sells", "Robert", x, "A")
            if new_fact not in self.facts:
                print(f"-> Derived: Sells(Robert, {x}, A) via Missile({x}) ∧ Owns(A, {x})")
                new_facts.add(new_fact)
        return new_facts

    # Rule 3: Enemy(x, America) => Hostile(x)
    def rule_enemy_is_hostile(self):
        new_facts = set()
        for fact in list(self.facts):
            if fact[0] == "Enemy" and fact[2] == "America":
                x = fact[1]
                new_fact = ("Hostile", x)
                if new_fact not in self.facts:
                    print(f"-> Derived: Hostile({x}) via Enemy({x}, America) => Hostile({x})")
                    new_facts.add(new_fact)
        return new_facts

    # Rule 4: American(p) ∧ Weapon(q) ∧ Sells(p, q, r) ∧ Hostile(r) => Criminal(p)
    def rule_criminal_definition(self):
        new_facts = set()
        # Filter facts for fast cross-referencing
        americans = {f[1] for f in self.facts if f[0] == "American"}
        weapons = {f[1] for f in self.facts if f[0] == "Weapon"}
        hostiles = {f[1] for f in self.facts if f[0] == "Hostile"}
        sells_records = [f for f in self.facts if f[0] == "Sells"]

        for p in americans:
            for q in weapons:
                for r in hostiles:
                    # Check if Sells(p, q, r) exists
                    if ("Sells", p, q, r) in sells_records:
                        new_fact = ("Criminal", p)
                        if new_fact not in self.facts:
                            print(f"-> Derived: Criminal({p}) via American({p}) ∧ Weapon({q}) ∧ Sells({p}, {q}, {r}) ∧ Hostile({r})")
                            new_facts.add(new_fact)
        return new_facts

    # Run the Forward Chaining loop until a goal is reached or no new facts can be derived
    def prove(self, goal):
        print("Initial Knowledge Base Facts:")
        for fact in self.facts:
            print(f" - {fact[0]}({', '.join(fact[1:])})")
        print("\n--- Starting Forward Chaining Deduction ---")

        while True:
            new_facts_in_iteration = set()

            # Evaluate all rules
            for rule in self.rules:
                derived = rule()
                new_facts_in_iteration.update(derived)

            # If no new facts were discovered, we reached a fixed point
            if not new_facts_in_iteration:
                print("\nNo more new facts can be inferred.")
                break

            # Add new facts to the knowledge base
            self.facts.update(new_facts_in_iteration)

            # Check if target goal is proven
            if goal in self.facts:
                print(f"\n[SUCCESS] Goal {goal[0]}({', '.join(goal[1:])}) has been successfully proven!")
                return True

        print(f"\n[FAILURE] Could not prove the goal.")
        return False

# Execute the proof
if __name__ == "__main__":
    kb = ForwardChainingKB()
    target_goal = ("Criminal", "Robert")
    kb.prove(target_goal)


Initial Knowledge Base Facts:
 - American(Robert)
 - Missile(T1)
 - Enemy(A, America)
 - Owns(A, T1)

--- Starting Forward Chaining Deduction ---
-> Derived: Weapon(T1) via Missile(T1) => Weapon(T1)
-> Derived: Sells(Robert, T1, A) via Missile(T1) ∧ Owns(A, T1)
-> Derived: Hostile(A) via Enemy(A, America) => Hostile(A)
-> Derived: Criminal(Robert) via American(Robert) ∧ Weapon(T1) ∧ Sells(Robert, T1, A) ∧ Hostile(A)

[SUCCESS] Goal Criminal(Robert) has been successfully proven!
